# Introducción a SmolAgents

Este notebook demuestra el uso de la librería **smolagents** de Hugging Face para crear agentes de IA que pueden:

- Ejecutar código Python de forma autónoma
- Buscar información en la web
- Utilizar diferentes modelos LLM como backend (DeepSeek, Claude en Bedrock)

## Tipos de agentes utilizados

| Agente | Descripción |
|--------|-------------|
| `CodeAgent` | Genera y ejecuta código Python para resolver tareas |
| `ToolCallingAgent` | Invoca herramientas directamente sin generar código intermedio |

## Herramientas (Tools) utilizadas

| Herramienta | Descripción |
|-------------|-------------|
| `DuckDuckGoSearchTool` | Búsqueda web vía DuckDuckGo |
| `PythonInterpreterTool` | Intérprete de Python para cálculos |
| `WebSearchTool` | Búsqueda web con acceso a contenido de páginas |

## Modelos LLM utilizados

| Modelo | Proveedor |
|--------|-----------|
| `deepseek-chat` | DeepSeek (vía LiteLLM) |
| `claude-opus-4-1` | Anthropic (vía Amazon Bedrock + LiteLLM) |

## 1. Instalación de dependencias

Se instalan los paquetes necesarios:
- `smolagents[toolkit]`: Librería principal con herramientas integradas
- `smolagents[litellm]`: Soporte para múltiples proveedores de LLM vía LiteLLM
- `boto3`: SDK de AWS para conectar con Amazon Bedrock

In [ ]:
!uv pip install 'smolagents[toolkit]'

In [ ]:
!uv pip install 'smolagents[litellm]'

In [ ]:
!uv pip install boto3

## 2. Verificación de la instalación

Se importa smolagents y se imprime la versión instalada para confirmar que todo está correcto.

In [ ]:
import smolagents
print(smolagents.__version__)

## 3. Importaciones y configuración del modelo

Se importan las clases principales y se configura el modelo **DeepSeek Chat** usando `LiteLLMModel`.

La API key se obtiene de los secrets de Google Colab (`userdata`).

In [ ]:
from smolagents import CodeAgent, LiteLLMModel, DuckDuckGoSearchTool, PythonInterpreterTool


In [ ]:
from google.colab import userdata


In [ ]:
model = LiteLLMModel(
    model_id="deepseek/deepseek-chat",
    api_key=userdata.get("DEEPSEEK_API_KEY"),
)

## 4. Ejemplo básico: CodeAgent sin herramientas

El `CodeAgent` puede resolver tareas generando y ejecutando código Python internamente, incluso sin herramientas externas.

**Tarea:** Calcular la suma de los números del 1 al 10.

In [ ]:
from smolagents import CodeAgent, InferenceClientModel


# Create an agent with no tools
agent = CodeAgent(tools=[], model=model)

# Run the agent with a task
result = agent.run("Calculate the sum of numbers from 1 to 10")
print(result)

## 5. CodeAgent con búsqueda web (DuckDuckGo)

Al agregar `DuckDuckGoSearchTool`, el agente puede buscar información actualizada en internet.

**Tarea:** Consultar el clima actual en París.

In [ ]:
from smolagents import CodeAgent, InferenceClientModel, DuckDuckGoSearchTool

agent = CodeAgent(
    tools=[DuckDuckGoSearchTool()],
    model=model,
)

# Now the agent can search the web!
result = agent.run("What is the current weather in Paris?")
print(result)

## 6. Búsqueda de noticias

Otro ejemplo de búsqueda web: obtener los titulares de noticias del día.

**Tarea:** Obtener los 3 principales titulares de noticias de hoy.

In [ ]:
from smolagents import CodeAgent, LiteLLMModel, DuckDuckGoSearchTool



agent = CodeAgent(
    tools=[DuckDuckGoSearchTool()],
    model=model,
)

result = agent.run("What are the top 3 news headlines today?")
print(result)

## 7. CodeAgent con PythonInterpreterTool

El `PythonInterpreterTool` permite al agente ejecutar código Python en un sandbox seguro para realizar cálculos complejos.

**Tarea:** Calcular el 10° número de Fibonacci.

In [ ]:
agent = CodeAgent(tools=[PythonInterpreterTool()], model=model)
response = agent.run("Calculate the 10th Fibonacci number.")
print(response)

## 8. Uso de Claude Opus 4.1 en Amazon Bedrock

Este ejemplo muestra cómo usar un modelo de Anthropic (Claude Opus 4.1) hospedado en **Amazon Bedrock** como backend del agente.

### Configuración:
- Se configura el token de autenticación de Bedrock desde los secrets de Colab
- Se usa `LiteLLMModel` con el prefijo `bedrock/converse/` para indicar el proveedor
- Región: `us-east-1`

**Tarea:** Investigar cuánto tarda un viaje en tren de Nueva York a Los Ángeles.

In [ ]:
import os
from smolagents import CodeAgent, LiteLLMModel, DuckDuckGoSearchTool
from google.colab import userdata

## Configurar token de Bedrock
os.environ["AWS_BEARER_TOKEN_BEDROCK"] = userdata.get("AWS_BEARER_TOKEN_BEDROCK")

# Usar Claude Opus 4.1 en Amazon Bedrock via LiteLLM


model_bedrock = LiteLLMModel(
    model_id="bedrock/converse/us.anthropic.claude-opus-4-1-20250805-v1:0",
    aws_region_name="us-east-1",
)

agent = CodeAgent(
    tools=[DuckDuckGoSearchTool()],
    model=model_bedrock,
)

result = agent.run("""
How long does it take to travel from
New York to Los Angeles by train?
""")
print(result)

## 9. ToolCallingAgent con WebSearchTool

A diferencia del `CodeAgent` que genera código, el `ToolCallingAgent` invoca herramientas directamente usando el formato nativo de "tool calling" del modelo.

Aquí se usa `WebSearchTool` que, además de buscar, puede acceder al contenido de URLs específicas.

**Tarea:** Obtener el título de la página del blog de Hugging Face.

In [ ]:
from smolagents import ToolCallingAgent, WebSearchTool

agent = ToolCallingAgent(tools=[WebSearchTool()], model=model_bedrock)
agent.run("Could you get me the title of the page at url 'https://huggingface.co/blog'?")

---

## Resumen

Este notebook cubre los conceptos fundamentales de **smolagents**:

1. **Instalación** con `uv pip` (toolkit, litellm, boto3)
2. **CodeAgent sin herramientas** — genera código para resolver tareas simples
3. **CodeAgent + DuckDuckGoSearchTool** — búsqueda web en tiempo real
4. **CodeAgent + PythonInterpreterTool** — ejecución de código en sandbox
5. **Integración con Amazon Bedrock** — uso de Claude Opus 4.1 como modelo
6. **ToolCallingAgent** — invocación directa de herramientas (sin generación de código)

### Requisitos previos
- API Key de DeepSeek (almacenada en secrets de Colab como `DEEPSEEK_API_KEY`)
- Token de AWS Bedrock (almacenado en secrets de Colab como `AWS_BEARER_TOKEN_BEDROCK`)
- Entorno Google Colab con GPU T4